# TMJDLC — Pyrogram-only Telegram Bot (Google Colab)

**Current runtime: Pyrogram MTProto only.** Pyrogram receives all updates, handles commands/callbacks, sends messages, and uploads videos/photos. No `python-telegram-bot`, Bot API `getUpdates`, or webhook is used. Neon PostgreSQL stores the durable queue and channel settings.

**Required Colab Secrets:** `TG_BOT_TOKEN`, `TG_API_ID`, `TG_API_HASH`, `TG_ADMIN_IDS`, `DATABASE_URL`.


## 1. Add Colab Secrets

- `TG_BOT_TOKEN` — token from @BotFather
- `TG_API_ID` — Telegram API ID from my.telegram.org
- `TG_API_HASH` — Telegram API hash from my.telegram.org
- `TG_ADMIN_IDS` — your numeric Telegram user ID
- `DATABASE_URL` — Neon connection string for `Javtiful-uploader`

Enable notebook access for all five secrets. Keep them private.


In [ ]:
#@title 2. Force-sync latest main and install dependencies
%cd /content
!if [ -d /content/TMJDLC/.git ]; then git -C /content/TMJDLC fetch origin main && git -C /content/TMJDLC reset --hard origin/main; else git clone --depth 1 --branch main https://github.com/Telebotfaroff/TMJDLC.git /content/TMJDLC; fi
%cd /content/TMJDLC
!python -m pip install --upgrade pip
!python -m pip install -e .
!sudo apt-get update -qq && sudo apt-get install -y ffmpeg
!python -m playwright install --with-deps chromium
!python -m compileall -q tmjdlc
import subprocess
print('Synced commit:', subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('Pyrogram-only source compiled successfully.')


In [ ]:
#@title 3. Load credentials and configure workers
import os
from google.colab import userdata
SECRET_NAMES = ['TG_BOT_TOKEN', 'TG_API_ID', 'TG_API_HASH', 'TG_ADMIN_IDS', 'DATABASE_URL']
missing=[]
for key in SECRET_NAMES:
    try: value=userdata.get(key)
    except Exception: value=None
    if not value: missing.append(key)
    else: os.environ[key]=str(value).strip()
if missing: raise RuntimeError('Missing Colab Secrets: '+', '.join(missing))
MAX_PARALLEL = 2 #@param {type:'slider', min:1, max:4, step:1}
os.environ['TMJDLC_MAX_PARALLEL']=str(MAX_PARALLEL)
os.environ['TMJDLC_WORK_DIR']='/content/tmjdlc-bot'
os.makedirs(os.environ['TMJDLC_WORK_DIR'],exist_ok=True)
print('Secrets loaded (hidden). Worker count:',MAX_PARALLEL)


## 4. Start the Pyrogram bot

After running the install and secrets cells, start the bot below. The bot uses Pyrogram MTProto for **both receiving updates and uploading large media**. If you already had an older notebook/runtime open, use **Runtime → Restart session**, rerun cells 2 and 3, then start the bot.


In [ ]:
#@title 5. Start TMJDLC — Pyrogram MTProto only
%cd /content/TMJDLC
!bash -o pipefail -c 'PYTHONUNBUFFERED=1 python -u -m tmjdlc.bot 2>&1 | tee -a /content/tmjdlc-bot/bot.log'
